---
title: Build a Qiskit Function for chemistry simulation
description: Learn how to deploy and execute the chemistry workflow template
---

# Build and run a Qiskit Function template for electronic structure simulation with an implicit solvent model

{/* cspell:ignore pvdz, fcisolver, avas, ncas, nelecas, ecore, chkfile, fcivec, hcore, ncore, myci, sqdvec, myeps, mymethod, mysolvmethod, myavas, mcscf, MCSCF, chkfile, prqs */}



This template, developed in collaboration with the Cleveland Clinic, consists of a workflow to calculate the ground state energy and solvation free energy of a molecule in an implicit solvent [\[1\]](#references). These simulations are based on the sample-based quantum diagonalization (SQD) method [\[2-6\]](#references) and the integral equation formalism polarizable continuum model (IEF-PCM) of solvent [\[7\]](#references).

For a step-by-step walkthrough of this same workflow with a worked example, see the [Implicit solvent calculations using Qiskit Serverless](/docs/tutorials/implicit-solvent-calculations) tutorial.

This guide utilizes the template with a methanol molecule as the solute, the electronic structure of which is simulated explicitly, and water as the solvent, approximated as a continuous dielectric medium. To account for the [electron correlation effects](https://onlinelibrary.wiley.com/doi/epdf/10.1002/ijch.202100111) in methanol, while maintaining the balance between the computational cost and accuracy, we only include the $\sigma$, $\sigma^{*}$, and lone pair orbitals in the active space simulated with SQD IEF-PCM. This orbital selection is done with [atomic valence active space (AVAS) method](https://github.com/pyscf/pyscf.github.io/blob/master/examples/mcscf/43-avas.py) using the C\[2s,2p], O\[2s,2p], and H\[1s] atomic orbital components, which results in the active space of 14 electrons and 12 orbitals (14e,12o). The reference orbitals are calculated with closed-shell Hartree Fock using the cc-pvdz basis set.



## Workflow introduction

This interactive guide shows how to upload this function template to Qiskit Serverless and run an example workload. The template is structured as a Qiskit pattern with four steps:

#### 1. Collect input and map the problem

This step takes the geometry of the molecule, selected active space, solvation model, LUCJ options, and SQD options as an input. It then produces the PySCF Checkpoint file, which contains the Hartree-Fock (HF) IEF-PCM data. This data will be used in the SQD portion of the workflow. For the LUCJ portion of the workflow, the input section also generates the gas-phase HF data, which is stored internally in PySCF FCIDUMP format.

The information from the HF gas-phase simulation and the definition of the active space are taken as input. Importantly, it also uses the user-defined information from the input section concerning the error suppression, number of shots, circuit transpiler optimization level, and the qubit layout.

It generates one-electron and two-electron integrals within the defined active space. The integrals are then used to perform classical CCSD calculations, which return t2 amplitudes that we use to parametrize the LUCJ circuit.

#### 2. Optimize the circuit

The LUCJ circuit is then transpiled into an ISA circuit for the target hardware. A Sampler primitive is then instantiated with a default set of error mitigation options to manage the execution.

#### 3. Execute the circuit

The LUCJ calculations return the bitstrings for each measurement, where these bitstrings correspond to electron configurations of the studied system. The bitstrings are then used as input for post-processing.

#### 4. Post-process by using SQD

This final step takes the PySCF Checkpoint file containing the HF IEF-PCM information, the bitstrings representing the electron configurations predicted by LUCJ, and the user-defined SQD options selected in the input section as input. As  output, it produces the SQD IEF-PCM total energy of the lowest energy batch and the corresponding solvation free energy.



### Options

For this template you must specify options for generating the LUCJ circuit, and SQD run parameters.

#### LUCJ options

When the LUCJ quantum circuit is executed, a set of samples that represent the computational basis states from the probability distribution of the molecular system are produced. To balance the depth of the LUCJ circuit and its expressibility, the qubits corresponding to the spin orbitals with the opposite spin have the two-qubit gates applied between them when these qubits are neighbors through a single ancilla qubit. To implement this approach on IBM hardware with a heavy-hex topology, qubits that represent the spin orbitals with the same spin are connected through a line topology where each line takes a zig-zag shape due to the heavy-hex connectivity of the target hardware, while the qubits that represent the spin orbitals with the opposite spin only have a connection at every fourth qubit.

<Accordion>
  <AccordionItem title="Click to expand for more details on the required options:">
    The user has to provide the `initial_layout` array corresponding to the qubits that satisfy this [*zig-zag* pattern](https://pubs.rsc.org/en/content/articlehtml/2023/sc/d3sc02516k) in the `lucj_options` section of the SQD IEF-PCM function. In case of SQD IEF-PCM (14e,12o)/cc-pvdz simulations of methanol, we chose the initial qubit layout corresponding to the main diagonal of the Eagle R3 QPU. Here, the first 12 elements of the `initial_layout` array `[0, 14, 18, 19, 20, 33, 39, 40, 41, 53, 60, 61, ...]` correspond to the alpha spin orbitals. The last 12 elements `[... 2, 3, 4, 15, 22, 23, 24, 34, 43, 44, 45, 54]` correspond to beta spin orbitals.

    Importantly, the user has to determine the `number_of_shots`, which corresponds to the number of measurements in the LUCJ circuit. The number of shots needs to be sufficiently large because the first step of S-CORE procedure relies on the samples in the right particle sector to obtain the initial approximation to the ground-state occupation number distribution.

    The number of shots is highly system- and hardware-dependent, but [non-covalent](https://arxiv.org/abs/2410.09209), [fragment-based](https://arxiv.org/abs/2411.09861), and [implicit solvent](https://pubs.acs.org/doi/10.1021/acs.jpcb.5c01030) SQD studies suggest that one can reach the chemical accuracy by following these guidelines:

    * 20,000 - 200,000 shots for systems with fewer than 16 molecular orbitals (32 spin orbitals)
    * 200,000 shots for systems with 16 - 18 molecular orbitals
    * 200,000 - 2,000,000 shots for systems with more than 18 molecular orbitals

    The required number of shots is affected by the number of spin orbitals in the studied system and by the size of the Hilbert space corresponding to the selected active space within the studied system. Generally, instances with smaller Hilbert spaces require fewer shots. Other available LUCJ options are [circuit transpiler optimization level](https://docs.quantum.ibm.com/guides/set-optimization) and [error suppression options](https://docs.quantum.ibm.com/guides/error-mitigation-and-suppression-techniques). Note that these options also affect the required number of shots and the resulting accuracy.
  </AccordionItem>
</Accordion>



#### SQD options

Important options in SQD simulations include the `sqd_iterations`, `number_of_batches`, and `samples_per_batch`. Generally, the lower number of samples per batch can be counteracted with more batches (`number_of_batches`) and more iterations of S-CORE (`sqd_iterations`). With more batches we can sample more variations of the configurational subspaces. Since the lowest-energy batch is taken as the solution for the ground state energy of the system, more batches can improve the results through better statistics. Additional iterations of S-CORE allow more configurations to be recovered from the original LUCJ distribution if the number of samples in the correct particle sector is low. This can allow the number of samples per batch to be reduced.

<Accordion>
  <AccordionItem title="Click to expand for more information about configuring the SQD options:">
    An alternative strategy is to use more samples per batch, which ensures that most of the initial LUCJ samples in right particle space are used during the S-CORE procedure, and individual subspaces encapsulate a sufficient variety of electron configurations. In turn, this reduces the number of required S-CORE steps, where only two or three iterations of SQD are needed if the number of samples per batch is large enough. However, more samples per batch results in a higher computational cost of each diagonalization step. Hence, the balance between the accuracy and computational cost in SQD simulations can be achieved by choosing  `sqd_iterations`, `number_of_batches`, and `samples_per_batch` optimally.

    The [SQD IEF-PCM study](https://pubs.acs.org/doi/10.1021/acs.jpcb.5c01030) shows that when three iterations of S-CORE are used, the chemical accuracy can be reached by following these guidelines:

    * 600 samples per batch in methanol SQD IEF-PCM (14e,12o) simulations
    * 1500 samples per batch in methylamine SQD IEF-PCM (14e,13o) simulations
    * 6000 samples per batch in water SQD IEF-PCM (8e,23o) simulations
    * 16000 samples per batch in ethanol SQD IEF-PCM (20e,18o) simulations

    Just like the required number of shots in LUCJ, the required number of samples per batch used in S-CORE procedure is highly system- and hardware-dependent. The examples above can be used to estimate the initial point for the benchmark of required number of samples per batch. The tutorial on systematic benchmark of the required number of samples per batch can be found [here](https://qiskit.github.io/qiskit-addon-sqd/how_tos/choose_subspace_dimension.html).
  </AccordionItem>
</Accordion>



## Deploy and execute the template SQD IEF-PCM function



### Authentication

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (token), which can be found on the [IBM Quantum Platform]() dashboard. This allows for the instantiation of the serverless client to upload or run the selected function:

```python
from qiskit_ibm_catalog import QiskitServerless

serverless = QiskitServerless(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    # For `token`, use the 44-character API_KEY you created
    # and saved from the IBM Quantum Platform Home dashboard
    token="YOUR_API_KEY"
)
```

Optionally, use `save_account()` to save your credentials in a local environment (see the [Set up your IBM Cloud account](/docs/guides/cloud-setup#cloud-save) guide). Note that this writes your credentials to the same file as [`QiskitRuntimeService.save_account()`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account):

```python
QiskitServerless.save_account(token="YOUR_API_KEY",
    channel="ibm_quantum_platform", instance="INSTANCE_CRN")
```

If the [account is saved](/docs/guides/save-credentials), there is no need to provide the token to authenticate:



In [ ]:
from qiskit_ibm_catalog import QiskitServerless

serverless = QiskitServerless()

### Upload the template



To upload a custom Qiskit Function, you must first instantiate a `QiskitFunction` object that defines the function source code. The title will allow you to identify the function once it's in the remote cluster. The main entry point is the file that contains `if __name__ == "__main__"`. If your workflow requires additional source files, you can define a working directory that will be uploaded together with the entry point.



In [ ]:
from qiskit_ibm_catalog import QiskitFunction

template = QiskitFunction(
    title="sqd_pcm_template",
    entrypoint="sqd_pcm_entrypoint.py",
    # all files in `working_dir` will be uploaded
    working_dir="./source_files/",
    dependencies=[
        "ffsim==0.0.54",
        "pyscf==2.9.0",
        "qiskit_addon_sqd==0.10.0",
    ],
)
print(template)

QiskitFunction(sqd_pcm_template)


Once the instance is ready, upload it to serverless:



In [3]:
serverless.upload(template)

QiskitFunction(sqd_pcm_template)

To check if the program successfully uploaded, use `serverless.list()`:



In [4]:
serverless.list()

[QiskitFunction(sqd_pcm_template),
 QiskitFunction(hamiltonian_simulation_template)]

## Load and run the template remotely



The function template has been uploaded, so you can run it remotely with Qiskit Serverless. First, load the template by name:



In [5]:
template = serverless.load("sqd_pcm_template")
print(template)

QiskitFunction(sqd_pcm_template)


Next, run the template with the domain-level inputs for SQD-IEF PCM. This example specifies a methanol-based workload.



In [ ]:
molecule = {
    "atom": """
    O -0.04559 -0.75076 -0.00000;
    C -0.04844 0.65398 -0.00000;
    H 0.85330 -1.05128 -0.00000;
    H -1.08779 0.98076 -0.00000;
    H 0.44171 1.06337 0.88811;
    H 0.44171 1.06337 -0.88811
    """,  # Must be specified
    "basis": "cc-pvdz",  # default is "sto-3g"
    "spin": 0,  # default is 0
    "charge": 0,  # default is 0
    "verbosity": 0,  # default is 0
    "number_of_active_orb": 12,  # Must be specified
    "number_of_active_alpha_elec": 7,  # Must be specified
    "number_of_active_beta_elec": 7,  # Must be specified
    "avas_selection": [
        "%d O %s" % (k, x) for k in [0] for x in ["2s", "2px", "2py", "2pz"]
    ]
    + ["%d C %s" % (k, x) for k in [1] for x in ["2s", "2px", "2py", "2pz"]]
    + ["%d H 1s" % k for k in [2, 3, 4, 5]],  # default is None
}

solvent_options = {
    # See https://manual.q-chem.com/5.4/topic_pcm-em.html for all methods
    "method": "IEF-PCM",  # other available methods are COSMO, C-PCM, SS(V)PE
    "eps": 78.3553,  # value for water
}

lucj_options = {
    "initial_layout": [
        0,
        14,
        18,
        19,
        20,
        33,
        39,
        40,
        41,
        53,
        60,
        61,
        2,
        3,
        4,
        15,
        22,
        23,
        24,
        34,
        43,
        44,
        45,
        54,
    ],
    "dynamical_decoupling_choice": True,
    "twirling_choice": True,
    "number_of_shots": 200000,
    "optimization_level": 2,
}

sqd_options = {
    "sqd_iterations": 3,
    "number_of_batches": 10,
    "samples_per_batch": 1000,
    "max_davidson_cycles": 200,
}

backend_name = "ibm_sherbrooke"

In [7]:
job = template.run(
    backend_name=backend_name,
    molecule=molecule,
    solvent_options=solvent_options,
    lucj_options=lucj_options,
    sqd_options=sqd_options,
)
print(job.job_id)

39f8fb70-79b2-43ca-b723-84e6b6135821


Check the detailed status of the job:



In [18]:
import time

t0 = time.time()
status = job.status()
if status == "QUEUED":
    print(f"time = {time.time()-t0:.2f}, status = QUEUED")
while True:
    status = job.status()
    if status == "QUEUED":
        continue
    print(f"time = {time.time()-t0:.2f}, status = {status}")
    if status == "DONE" or status == "ERROR":
        break

time = 2.35, status = DONE


While the job is running, you can fetch logs created from the `logger.info` outputs. These can provide actionable information about the progress of the SQD IEF-PCM workflow. For example, the same spin orbital connections, or the two-qubit depth of the final ISA circuit intended for execution on hardware.



In [ ]:
print(job.logs())

Calling for the job result blocks the rest of the program until a result is available. After the job is done, you can retrieve the results. These include the solvation free energy, as well as information about the lowest energy batch, lowest energy value, and other useful information such as the total solver duration.



In [16]:
result = job.result()

result

{'total_energy_hist': array([[-115.14768518, -115.1368396 , -114.19181692, -115.13745429,
         -115.1445012 , -114.19673326, -115.1547003 , -114.20563866,
         -115.13748344, -115.14764974],
        [-115.15768392, -115.15850126, -115.15857275, -115.15770916,
         -115.15801684, -115.15822125, -115.15833521, -115.15844051,
         -115.15735538, -115.15862354],
        [-115.15795148, -115.15847925, -115.15856677, -115.15811156,
         -115.15815602, -115.15785171, -115.1583672 , -115.1585533 ,
         -115.15833528, -115.15808791]]),
 'spin_squared_value_hist': array([[5.37327508e-03, 1.32981759e-02, 1.36214922e-02, 8.84413615e-03,
         7.26723578e-03, 1.94875195e-02, 3.03153152e-03, 6.07543106e-03,
         1.04951849e-02, 5.36529204e-03],
        [6.39397528e-04, 1.36814350e-04, 9.09054260e-05, 5.99361358e-04,
         3.64261739e-04, 2.54905866e-04, 2.32540370e-04, 1.53181990e-04,
         7.23519739e-04, 6.80737671e-05],
        [4.53776416e-04, 1.63043449e-04,

Note that the result metadata includes a resource usage summary that lets you better estimate the QPU and CPU time required for each workload (this example ran on a dummy device, so actual resource usage times might differ).



After the job completes, the entire logging output will be available.



In [17]:
print(job.logs())

2025-06-27 08:42:41,358	INFO job_manager.py:531 -- Runtime env is setting up.
sqd_pcm_entrypoint.run_function:INFO:2025-06-27 08:42:45,015: Starting runtime service
sqd_pcm_entrypoint.run_function:INFO:2025-06-27 08:42:45,621: Backend: ibm_sherbrooke
sqd_pcm_entrypoint.run_function:INFO:2025-06-27 08:42:46,809: Initializing molecule object
sqd_pcm_entrypoint.run_function:INFO:2025-06-27 08:42:51,599: Performing CCSD
Parsing /tmp/ray/session_2025-06-27_08-42-13_898146_1/runtime_resources/working_dir_files/_ray_pkg_4bc93dcc58c04b91/output_sqd_pcm/2025-06-27_08-42-45.fcidump.txt
Overwritten attributes  get_ovlp get_hcore  of <class 'pyscf.scf.hf_symm.SymAdaptedRHF'>
/usr/local/lib/python3.11/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)
/usr/local/lib/python3.11/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute intor_symmetric because it is not JS

## Cite this project

If you use the SQD IEF-PCM template in your research, please refer to the [repository](https://github.com/qiskit-community/qiskit-function-templates/tree/main/chemistry/sqd_pcm#citing-this-project) for detailed citation information.



## Next steps

<Admonition type="info" title="Recommendations">
  * Follow the [Implicit solvent calculations using Qiskit Serverless](/docs/tutorials/implicit-solvent-calculations) tutorial for a worked example of this same workflow
  * Review the guide on building a function template for [Hamiltonian simulation](/docs/guides/function-template-hamiltonian-simulation)
  * Check out the source files for this template on [GitHub](https://github.com/qiskit-community/qiskit-function-templates/tree/main/chemistry/sqd_pcm)
</Admonition>



#### References

\[1] Danil Kaliakin, Akhil Shajan, Fangchun Liang, and Kenneth M. Merz Jr. [Implicit Solvent Sample-Based Quantum Diagonalization](https://pubs.acs.org/doi/10.1021/acs.jpcb.5c01030), The Journal of Physical Chemistry B, 2025, DOI: 10.1021/acs.jpcb.5c01030

\[2] Javier Robledo-Moreno, et al., [Chemistry Beyond Exact Solutions on a Quantum-Centric Supercomputer](https://arxiv.org/abs/2405.05068), arXiv:2405.05068 \[quant-ph].

\[3] Jeffery Yu, et al., [Quantum-Centric Algorithm for Sample-Based Krylov Diagonalization](https://arxiv.org/abs/2501.09702), arXiv:2501.09702 \[quant-ph].

\[4] Keita Kanno, et al., [Quantum-Selected Configuration Interaction: classical diagonalization of Hamiltonians in subspaces selected by quantum computers](https://arxiv.org/abs/2302.11320), arXiv:2302.11320 \[quant-ph].

\[5] Kenji Sugisaki, et al., [Hamiltonian simulation-based quantum-selected configuration interaction for large-scale electronic structure calculations with a quantum computer](https://arxiv.org/abs/2412.07218), arXiv:2412.07218 \[quant-ph].

\[6] Mathias Mikkelsen, Yuya O. Nakagawa, [Quantum-selected configuration interaction with time-evolved state](https://arxiv.org/abs/2412.13839), arXiv:2412.13839 \[quant-ph].

\[7] Herbert, John M. [Dielectric continuum methods for quantum chemistry. WIREs Computational Molecular Science](https://wires.onlinelibrary.wiley.com/doi/10.1002/wcms.1519), 2021, 11, 1759-0876.

\[8] Saki, A. A.; Barison, S.; Fuller, B.; Garrison, J. R.; Glick, J. R.; Johnson, C.; Mezzacapo, A.; Robledo-Moreno, J.; Rossmannek, M.; Schweigert, P. et al. Qiskit addon: sample-based quantum diagonalization, 2024; [https://github.com/Qiskit/qiskit-addon-sqd](https://github.com/Qiskit/qiskit-addon-sqd)

\[9] Asun, Q.; Zhang, X.; Banerjee, S.; Bao, P.; Barbry, M.; Blunt, N. S.; Bogdanov, N. A.; Booth, G. H.; Chen, J.; Cui, Z.-H. PySCF: Python-based Simulations of Chemistry Framework, 2025; [https://github.com/pyscf/pyscf](https://github.com/pyscf/pyscf)

\[10] Kevin J. Sung; et al., FFSIM: Faster simulations of fermionic quantum circuits, 2024. [https://github.com/qiskit-community/ffsim](https://github.com/qiskit-community/ffsim)



© IBM Corp., 2017-2026